# Stage 5: Ablation Studies & Inference Quantum Efficiency Benchmarks

**Project Title:** Adaptive Hybrid Quantum-Classical Neural Network for Medical Disease Prediction Using Dynamic Quantum Layer Selection and Explainable AI  
**Focus:** Full Multi-Model Comparison (M1 vs. M2 vs. M3 vs. M4), Inference-Time Quantum Efficiency Accounting, Publication Tables 2 & 3, and Tradeoff Visualizations

---

## 1. Stage Overview & Multi-Model Comparison
In this stage, we execute the formal ablation study and computational efficiency benchmarking across all three clinical datasets.

### Models Evaluated in Rigorous Comparison:
1. **M1 — Classical MLP**: Fixed 2-hidden-layer feedforward classical benchmark.
2. **M2 — Fixed HQNN**: Conventional hybrid quantum-classical network executing $2$ fixed quantum variational layers for every sample.
3. **M3 — Adaptive HQNN (Proposed)**: Dynamic quantum-layer selection allocating $0$, $1$, or $2$ quantum layers based on instance confidence.
4. **M4 — Adaptive Classical Control (Ablation)**: Adaptive dynamic computational depth using purely classical refinement modules (no quantum circuits). Isolates whether benefits arise from adaptive computation in general vs. quantum variational processing specifically.

### Quantum Efficiency Accounting Principles:
* **Training Workload**: Multi-path joint training evaluates all paths during gradient optimization.
* **Inference Workload**: Dynamic routing executes only the designated quantum path ($0$, $1$, or $2$ Q-layers), drastically reducing inference-time circuit evaluations.
* **Workload Representation**: All quantum metrics represent *simulated gate/circuit evaluations* (estimated quantum workload), not physical QPU costs.


## 2. Environment Setup & Dependency Installation


In [1]:
# Install dependencies if running in Google Colab
# !pip install -q pennylane torch scikit-learn pandas numpy matplotlib seaborn tabulate

import os
import sys
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Global visualization settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Deterministic Seed Configuration
RANDOM_STATE = 42

def set_deterministic_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_deterministic_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Execution device: {device}")
print(f"Deterministic seed locked to: {RANDOM_STATE}")


PyTorch version: 2.6.0+cpu
Execution device: cpu
Deterministic seed locked to: 42


## 3. Project Directory Verification & Path Management


In [2]:
# Determine workspace root
if os.path.exists("notebooks") or os.path.exists("data"):
    PROJECT_DIR = os.path.abspath(".")
elif os.path.exists("../data") or os.path.basename(os.getcwd()) == 'notebooks':
    PROJECT_DIR = os.path.abspath("..")
elif os.path.exists(r"C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn"):
    PROJECT_DIR = r"C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
DATA_SPLITS_DIR = os.path.join(PROJECT_DIR, 'data', 'splits')
RESULTS_METRICS_DIR = os.path.join(PROJECT_DIR, 'results', 'metrics')
RESULTS_MODELS_DIR = os.path.join(PROJECT_DIR, 'results', 'models')
RESULTS_FIGURES_DIR = os.path.join(PROJECT_DIR, 'results', 'figures')
RESULTS_TABLES_DIR = os.path.join(PROJECT_DIR, 'results', 'tables')

print("Project directories verified at:", PROJECT_DIR)


Project directories verified at: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn


## 4. Dataset Loader & Fold Preprocessor


In [3]:
def load_dataset_data(dataset_key):
    """Load raw dataset and split configuration for a given dataset key."""
    split_path = os.path.join(DATA_SPLITS_DIR, dataset_key, "splits_config.json")
    with open(split_path, "r") as f:
        split_info = json.load(f)
        
    if dataset_key == "breast_cancer":
        raw_file = os.path.join(DATA_RAW_DIR, "breast_cancer", "wdbc.data")
        feature_names = [
            'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
            'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
            'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
            'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se',
            'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
            'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst'
        ]
        raw_cols = ['id', 'diagnosis'] + feature_names
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols)
        df = df_raw.drop(columns=['id']).copy()
        df['target'] = df['diagnosis'].map({'B': 0, 'M': 1}).astype(int)
        df = df.drop(columns=['diagnosis'])
        
    elif dataset_key == "heart":
        raw_file = os.path.join(DATA_RAW_DIR, "heart", "processed.cleveland.data")
        feature_names = [
            'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
            'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
        ]
        raw_cols = feature_names + ['num']
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols, na_values='?')
        df = df_raw.copy()
        df['target'] = (df['num'] > 0).astype(int)
        df = df.drop(columns=['num'])
        
    elif dataset_key == "diabetes":
        raw_file = os.path.join(DATA_RAW_DIR, "diabetes", "pima-indians-diabetes.csv")
        feature_names = [
            'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
        ]
        raw_cols = feature_names + ['Outcome']
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols)
        df = df_raw.copy()
        df['target'] = df['Outcome'].astype(int)
        df = df.drop(columns=['Outcome'])
        for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
            df[col] = df[col].replace(0, np.nan)
            
    return df, feature_names, split_info

class FoldPreprocessor:
    """Leakage-safe fold transformer fitting strictly on training feature matrix."""
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        
    def fit(self, X_train):
        self.imputer.fit(X_train)
        X_imp = self.imputer.transform(X_train)
        self.scaler.fit(X_imp)
        return self
        
    def transform(self, X):
        X_imp = self.imputer.transform(X)
        return self.scaler.transform(X_imp)
        
    def fit_transform(self, X_train):
        return self.fit(X_train).transform(X_train)

print("Dataset loader ready.")


Dataset loader ready.


## 5. Model 4: Adaptive Classical Control Model (Ablation)

To test whether benefits stem from **adaptive computing in general** vs. **quantum variational processing specifically**, we implement **Model 4: Adaptive Classical Network**.


In [4]:
class AdaptiveClassicalControl(nn.Module):
    """
    Adaptive Classical Control Network (Model 4).
    Uses dynamic computational depth with purely classical layers (0 Q-layers).
    """
    def __init__(self, input_dim):
        super(AdaptiveClassicalControl, self).__init__()
        self.input_dim = input_dim
        
        # Classical Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.LayerNorm(16),
            nn.ReLU()
        )
        
        # Preliminary Head
        self.prelim_head = nn.Linear(16, 1)
        
        # Classical Refinement 1 (equivalent capacity to 1 Q-Layer)
        self.c1_refine = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
        # Classical Refinement 2 (equivalent capacity to 2 Q-Layers)
        self.c2_refine = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward_all_paths(self, x):
        h = self.encoder(x)
        p_logits = self.prelim_head(h).squeeze(-1)
        c1_logits = (p_logits + self.c1_refine(h).squeeze(-1))
        c2_logits = (p_logits + self.c2_refine(h).squeeze(-1))
        return p_logits, c1_logits, c2_logits
        
    def forward_adaptive(self, x, tau_high=0.85, tau_low=0.65):
        h = self.encoder(x)
        p_logits = self.prelim_head(h).squeeze(-1)
        p_probs = torch.sigmoid(p_logits)
        conf = torch.maximum(p_probs, 1.0 - p_probs)
        
        N = len(x)
        final_logits = torch.zeros(N, device=x.device)
        routes = np.zeros(N, dtype=int)
        
        for i in range(N):
            conf_i = conf[i].item()
            if conf_i >= tau_high:
                final_logits[i] = p_logits[i]
                routes[i] = 0
            elif conf_i >= tau_low:
                final_logits[i] = (p_logits[i] + self.c1_refine(h[i:i+1]).squeeze(-1))
                routes[i] = 1
            else:
                final_logits[i] = (p_logits[i] + self.c2_refine(h[i:i+1]).squeeze(-1))
                routes[i] = 2
                
        return final_logits, p_logits, conf, routes

print("AdaptiveClassicalControl (M4) defined.")


AdaptiveClassicalControl (M4) defined.


## 6. Leakage-Safe Nested 5-Fold Cross-Validation for Model 4 (Adaptive Classical Control)


In [5]:
def compute_medical_metrics(y_true, y_probs, threshold=0.5):
    """Compute medical metrics."""
    y_true = np.array(y_true)
    y_probs = np.array(y_probs)
    y_pred = (y_probs >= threshold).astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    try:
        auc = roc_auc_score(y_true, y_probs)
    except Exception:
        auc = 0.5
        
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": spec,
        "f1": f1,
        "roc_auc": auc,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }

def train_adaptive_classical_model(model, train_loader, val_loader, pos_weight, lr=0.005, weight_decay=1e-4, max_epochs=70, patience=12):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(max_epochs):
        model.train()
        train_loss_acc = 0.0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            p_logits, c1_logits, c2_logits = model.forward_all_paths(batch_x)
            
            loss_p = criterion(p_logits, batch_y)
            loss_c1 = criterion(c1_logits, batch_y)
            loss_c2 = criterion(c2_logits, batch_y)
            
            total_loss = loss_p + 0.5 * loss_c1 + 0.5 * loss_c2
            total_loss.backward()
            optimizer.step()
            train_loss_acc += total_loss.item() * len(batch_y)
            
        # Validation if provided
        if val_loader is not None and len(val_loader) > 0:
            model.eval()
            val_loss_acc = 0.0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    p_logits, c1_logits, c2_logits = model.forward_all_paths(batch_x)
                    
                    loss_p = criterion(p_logits, batch_y)
                    loss_c1 = criterion(c1_logits, batch_y)
                    loss_c2 = criterion(c2_logits, batch_y)
                    val_loss_acc += (loss_p + 0.5 * loss_c1 + 0.5 * loss_c2).item() * len(batch_y)
                    
            val_loss = val_loss_acc / len(val_loader.dataset)
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
                    
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def optimize_thresholds_classical_inner_cv(X_train_raw, y_train, input_dim, random_seed=42):
    inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_seed)
    oof_p_probs = np.zeros(len(y_train))
    oof_c1_probs = np.zeros(len(y_train))
    oof_c2_probs = np.zeros(len(y_train))
    
    for in_fold, (in_train_idx, in_val_idx) in enumerate(inner_skf.split(X_train_raw, y_train)):
        in_preproc = FoldPreprocessor()
        X_in_train = in_preproc.fit_transform(X_train_raw[in_train_idx])
        X_in_val = in_preproc.transform(X_train_raw[in_val_idx])
        y_in_train = y_train[in_train_idx]
        y_in_val = y_train[in_val_idx]
        
        n_pos = np.sum(y_in_train == 1)
        n_neg = np.sum(y_in_train == 0)
        pos_weight = float(n_neg / n_pos) if n_pos > 0 else 1.0
        
        in_train_loader = DataLoader(
            TensorDataset(torch.tensor(X_in_train, dtype=torch.float32), torch.tensor(y_in_train, dtype=torch.float32)),
            batch_size=32, shuffle=True
        )
        in_val_loader = DataLoader(
            TensorDataset(torch.tensor(X_in_val, dtype=torch.float32), torch.tensor(y_in_val, dtype=torch.float32)),
            batch_size=32, shuffle=False
        )
        
        set_deterministic_seed(random_seed + in_fold * 10)
        in_model = AdaptiveClassicalControl(input_dim=input_dim)
        in_model = train_adaptive_classical_model(
            in_model, in_train_loader, in_val_loader, pos_weight=pos_weight,
            lr=0.005, weight_decay=1e-4, max_epochs=50, patience=8
        )
        
        in_model.eval()
        with torch.no_grad():
            val_t = torch.tensor(X_in_val, dtype=torch.float32).to(device)
            p_logits, c1_logits, c2_logits = in_model.forward_all_paths(val_t)
            oof_p_probs[in_val_idx] = torch.sigmoid(p_logits).cpu().numpy()
            oof_c1_probs[in_val_idx] = torch.sigmoid(c1_logits).cpu().numpy()
            oof_c2_probs[in_val_idx] = torch.sigmoid(c2_logits).cpu().numpy()
            
    oof_conf = np.maximum(oof_p_probs, 1.0 - oof_p_probs)
    
    tau_high_grid = [0.75, 0.80, 0.85, 0.90, 0.95]
    tau_low_grid = [0.55, 0.60, 0.65, 0.70]
    
    best_score = -float('inf')
    best_tau_high = 0.85
    best_tau_low = 0.65
    
    for th in tau_high_grid:
        for tl in tau_low_grid:
            if th <= tl:
                continue
            sim_probs = np.where(oof_conf >= th, oof_p_probs, np.where(oof_conf >= tl, oof_c1_probs, oof_c2_probs))
            sim_c = np.where(oof_conf >= th, 0, np.where(oof_conf >= tl, 1, 2))
            avg_c = np.mean(sim_c)
            
            sim_preds = (sim_probs >= 0.5).astype(int)
            f1 = f1_score(y_train, sim_preds, zero_division=0)
            score = f1 - 0.05 * (avg_c / 2.0)
            
            if score > best_score:
                best_score = score
                best_tau_high = th
                best_tau_low = tl
                
    return best_tau_high, best_tau_low

m4_cv_results = []

for ds_key in ['breast_cancer', 'heart', 'diabetes']:
    set_deterministic_seed(RANDOM_STATE)
    df, feature_names, split_info = load_dataset_data(ds_key)
    ds_name = split_info["display_name"]
    print(f"\nEvaluating M4 Adaptive Classical on: {ds_name}")
    
    dev_idx = np.array(split_info["dev_indices"])
    test_idx = np.array(split_info["test_indices"])
    
    X_dev = df.iloc[dev_idx][feature_names].values
    y_dev = df.iloc[dev_idx]['target'].values
    X_test = df.iloc[test_idx][feature_names].values
    y_test = df.iloc[test_idx]['target'].values
    
    fold_models = []
    fold_preprocs = []
    fold_thresholds = []
    
    for fold_info in split_info["cv_folds"]:
        fold_num = fold_info["fold"]
        train_sub_idx = np.array(fold_info["train_dev_idx"])
        val_sub_idx = np.array(fold_info["val_dev_idx"])
        
        X_train_raw = X_dev[train_sub_idx]
        y_train = y_dev[train_sub_idx]
        X_val_raw = X_dev[val_sub_idx]
        y_val = y_dev[val_sub_idx]
        
        # 1. Inner CV threshold tuning
        tau_high_opt, tau_low_opt = optimize_thresholds_classical_inner_cv(
            X_train_raw, y_train, input_dim=len(feature_names), random_seed=RANDOM_STATE + fold_num * 100
        )
        fold_thresholds.append((tau_high_opt, tau_low_opt))
        
        # 2. Retrain outer model
        preprocessor = FoldPreprocessor()
        X_train_scaled = preprocessor.fit_transform(X_train_raw)
        X_val_scaled = preprocessor.transform(X_val_raw)
        
        n_pos = np.sum(y_train == 1)
        n_neg = np.sum(y_train == 0)
        pos_weight = float(n_neg / n_pos) if n_pos > 0 else 1.0
        
        train_ds = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        
        set_deterministic_seed(RANDOM_STATE + fold_num)
        model = AdaptiveClassicalControl(input_dim=len(feature_names))
        
        t0 = time.time()
        model = train_adaptive_classical_model(model, train_loader, val_loader=None, pos_weight=pos_weight)
        elapsed = time.time() - t0
        
        # 3. Apply frozen thresholds to untouched validation fold
        model.eval()
        with torch.no_grad():
            val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
            val_logits, _, _, _ = model.forward_adaptive(val_tensor, tau_high=tau_high_opt, tau_low=tau_low_opt)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
            
        metrics = compute_medical_metrics(y_val, val_probs)
        metrics["dataset"] = ds_key
        metrics["dataset_name"] = ds_name
        metrics["model"] = "Adaptive Classical (M4)"
        metrics["fold"] = fold_num
        metrics["runtime_sec"] = elapsed
        metrics["avg_quantum_layers"] = 0.00
        m4_cv_results.append(metrics)
        
        fold_models.append(model)
        fold_preprocs.append(preprocessor)
        
    # Test Ensemble Evaluation
    test_p_all = []
    test_c1_all = []
    test_c2_all = []
    
    for m, prep in zip(fold_models, fold_preprocs):
        X_test_scaled_f = prep.transform(X_test)
        test_t = torch.tensor(X_test_scaled_f, dtype=torch.float32).to(device)
        m.eval()
        with torch.no_grad():
            p_l, c1_l, c2_l = m.forward_all_paths(test_t)
            test_p_all.append(torch.sigmoid(p_l).cpu().numpy())
            test_c1_all.append(torch.sigmoid(c1_l).cpu().numpy())
            test_c2_all.append(torch.sigmoid(c2_l).cpu().numpy())
            
    ens_p = np.mean(test_p_all, axis=0)
    ens_c1 = np.mean(test_c1_all, axis=0)
    ens_c2 = np.mean(test_c2_all, axis=0)
    ens_conf = np.maximum(ens_p, 1.0 - ens_p)
    
    th_m4 = float(np.median([th for th, tl in fold_thresholds]))
    tl_m4 = float(np.median([tl for th, tl in fold_thresholds]))
    
    test_routes_m4 = np.where(ens_conf >= th_m4, 0, np.where(ens_conf >= tl_m4, 1, 2))
    ens_test = np.where(test_routes_m4 == 0, ens_p, np.where(test_routes_m4 == 1, ens_c1, ens_c2))
    
    test_metrics = compute_medical_metrics(y_test, ens_test)
    test_metrics["dataset"] = ds_key
    test_metrics["dataset_name"] = ds_name
    test_metrics["model"] = "Adaptive Classical (M4) - Test Ensemble"
    test_metrics["fold"] = "TEST_ENSEMBLE"
    test_metrics["avg_quantum_layers"] = 0.00
    m4_cv_results.append(test_metrics)

df_m4_all = pd.DataFrame(m4_cv_results)
df_m4_all.to_csv(os.path.join(RESULTS_METRICS_DIR, "m4_adaptive_classical_metrics.csv"), index=False)
print("[OK] M4 Adaptive Classical evaluation complete and saved.")


Evaluating M4 Adaptive Classical on: Wisconsin Diagnostic Breast Cancer (WDBC)
Evaluating M4 Adaptive Classical on: UCI Heart Disease (Cleveland)
Evaluating M4 Adaptive Classical on: Pima Indians Diabetes Database
[OK] M4 Adaptive Classical evaluation complete and saved.


## 7. Publication Table 2: Comprehensive Model Performance Comparison (Separating CV and Test Ensemble)


In [6]:
df_m1 = pd.read_csv(os.path.join(RESULTS_METRICS_DIR, "m1_classical_mlp_metrics.csv"))
df_m2 = pd.read_csv(os.path.join(RESULTS_METRICS_DIR, "m2_fixed_hqnn_metrics.csv"))
df_m3 = pd.read_csv(os.path.join(RESULTS_METRICS_DIR, "m3_adaptive_hqnn_metrics.csv"))
df_m4 = pd.read_csv(os.path.join(RESULTS_METRICS_DIR, "m4_adaptive_classical_metrics.csv"))

models_dict = {
    "Classical MLP (M1)": df_m1,
    "Fixed HQNN (M2)": df_m2,
    "Adaptive HQNN (M3)": df_m3,
    "Adaptive Classical (M4)": df_m4
}

table2_rows = []

for ds_key in ['breast_cancer', 'heart', 'diabetes']:
    ds_display = {
        'breast_cancer': 'Wisconsin Diagnostic Breast Cancer (WDBC)',
        'heart': 'UCI Heart Disease (Cleveland)',
        'diabetes': 'Pima Indians Diabetes'
    }[ds_key]
    
    for m_name, df_m in models_dict.items():
        df_cv = df_m[(df_m['dataset'] == ds_key) & (df_m['fold'] != 'TEST_ENSEMBLE')]
        df_test = df_m[(df_m['dataset'] == ds_key) & (df_m['fold'] == 'TEST_ENSEMBLE')]
        
        acc_cv = f"{df_cv['accuracy'].mean()*100:.2f} ± {df_cv['accuracy'].std()*100:.2f}%"
        f1_cv = f"{df_cv['f1'].mean()*100:.2f} ± {df_cv['f1'].std()*100:.2f}%"
        auc_cv = f"{df_cv['roc_auc'].mean():.4f} ± {df_cv['roc_auc'].std():.4f}"
        
        acc_test = f"{df_test['accuracy'].iloc[0]*100:.2f}%"
        f1_test = f"{df_test['f1'].iloc[0]*100:.2f}%"
        auc_test = f"{df_test['roc_auc'].iloc[0]:.4f}"
        
        table2_rows.append({
            "Dataset": ds_display,
            "Model": m_name,
            "CV Accuracy (%)": acc_cv,
            "CV F1-Score (%)": f1_cv,
            "CV ROC-AUC": auc_cv,
            "Holdout Test Accuracy (%)": acc_test,
            "Holdout Test F1-Score (%)": f1_test,
            "Holdout Test ROC-AUC": auc_test
        })

df_table2 = pd.DataFrame(table2_rows)

table2_csv_path = os.path.join(RESULTS_TABLES_DIR, "table2_model_performance_comparison.csv")
table2_md_path = os.path.join(RESULTS_TABLES_DIR, "table2_model_performance_comparison.md")
table2_tex_path = os.path.join(RESULTS_TABLES_DIR, "table2_model_performance_comparison.tex")

df_table2.to_csv(table2_csv_path, index=False)
with open(table2_md_path, "w", encoding="utf-8") as f:
    f.write("# Table 2: Comprehensive Model Performance Comparison across Clinical Benchmarks\n\n")
    try:
        f.write(df_table2.to_markdown(index=False))
    except Exception:
        f.write(df_table2.to_string(index=False))

latex_t2 = df_table2.to_latex(index=False, caption="Comprehensive performance comparison separating 5-fold cross-validation and holdout test evaluation.", label="tab:model_performance")
with open(table2_tex_path, "w", encoding="utf-8") as f:
    f.write(latex_t2)

print("[OK] Table 2 successfully exported to CSV, Markdown, and LaTeX.")
display(df_table2)


[OK] Table 2 successfully exported to CSV, Markdown, and LaTeX.
                                      Dataset  ... Holdout Test ROC-AUC
0   Wisconsin Diagnostic Breast Cancer (WDBC)  ...               0.9907
1   Wisconsin Diagnostic Breast Cancer (WDBC)  ...               0.9940
2   Wisconsin Diagnostic Breast Cancer (WDBC)  ...               0.9907
3   Wisconsin Diagnostic Breast Cancer (WDBC)  ...               0.9927
4               UCI Heart Disease (Cleveland)  ...               0.9578
5               UCI Heart Disease (Cleveland)  ...               0.9307
6               UCI Heart Disease (Cleveland)  ...               0.9524
7               UCI Heart Disease (Cleveland)  ...               0.9448
8                       Pima Indians Diabetes  ...               0.8148
9                       Pima Indians Diabetes  ...               0.7876
10                      Pima Indians Diabetes  ...               0.8144
11                      Pima Indians Diabetes  ...               0.8057


## 8. Publication Table 3: Estimated Inference Quantum Workload & Computational Efficiency


In [7]:
table3_rows = [
    {
        "Dataset": "Wisconsin Diagnostic Breast Cancer",
        "Model": "Fixed HQNN (M2)",
        "Avg Inference Q-Layers/Sample": "2.00",
        "0-Q %": "0.0%",
        "1-Q %": "0.0%",
        "2-Q %": "100.0%",
        "Inference Q-Reduction (%)": "0.0%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    },
    {
        "Dataset": "Wisconsin Diagnostic Breast Cancer",
        "Model": "Adaptive HQNN (M3)",
        "Avg Inference Q-Layers/Sample": f"{df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean():.2f} ± {df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].std():.2f}",
        "0-Q %": f"{df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['classical_only_pct'].mean():.1f}%",
        "1-Q %": f"{df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['one_qlayer_pct'].mean():.1f}%",
        "2-Q %": f"{df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['two_qlayers_pct'].mean():.1f}%",
        "Inference Q-Reduction (%)": f"{df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['quantum_reduction_pct'].mean():.1f}%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    },
    {
        "Dataset": "UCI Heart Disease",
        "Model": "Fixed HQNN (M2)",
        "Avg Inference Q-Layers/Sample": "2.00",
        "0-Q %": "0.0%",
        "1-Q %": "0.0%",
        "2-Q %": "100.0%",
        "Inference Q-Reduction (%)": "0.0%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    },
    {
        "Dataset": "UCI Heart Disease",
        "Model": "Adaptive HQNN (M3)",
        "Avg Inference Q-Layers/Sample": f"{df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean():.2f} ± {df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].std():.2f}",
        "0-Q %": f"{df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['classical_only_pct'].mean():.1f}%",
        "1-Q %": f"{df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['one_qlayer_pct'].mean():.1f}%",
        "2-Q %": f"{df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['two_qlayers_pct'].mean():.1f}%",
        "Inference Q-Reduction (%)": f"{df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['quantum_reduction_pct'].mean():.1f}%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    },
    {
        "Dataset": "Pima Indians Diabetes",
        "Model": "Fixed HQNN (M2)",
        "Avg Inference Q-Layers/Sample": "2.00",
        "0-Q %": "0.0%",
        "1-Q %": "0.0%",
        "2-Q %": "100.0%",
        "Inference Q-Reduction (%)": "0.0%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    },
    {
        "Dataset": "Pima Indians Diabetes",
        "Model": "Adaptive HQNN (M3)",
        "Avg Inference Q-Layers/Sample": f"{df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean():.2f} ± {df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].std():.2f}",
        "0-Q %": f"{df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['classical_only_pct'].mean():.1f}%",
        "1-Q %": f"{df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['one_qlayer_pct'].mean():.1f}%",
        "2-Q %": f"{df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['two_qlayers_pct'].mean():.1f}%",
        "Inference Q-Reduction (%)": f"{df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['quantum_reduction_pct'].mean():.1f}%",
        "Qubits": 4,
        "Circuit Depth": 11,
        "Gates / Sample": 17
    }
]

df_table3 = pd.DataFrame(table3_rows)

table3_csv_path = os.path.join(RESULTS_TABLES_DIR, "table3_efficiency_comparison.csv")
table3_md_path = os.path.join(RESULTS_TABLES_DIR, "table3_efficiency_comparison.md")
table3_tex_path = os.path.join(RESULTS_TABLES_DIR, "table3_efficiency_comparison.tex")

df_table3.to_csv(table3_csv_path, index=False)
with open(table3_md_path, "w", encoding="utf-8") as f:
    f.write("# Table 3: Estimated Inference Quantum Workload & Computational Efficiency\n\n")
    try:
        f.write(df_table3.to_markdown(index=False))
    except Exception:
        f.write(df_table3.to_string(index=False))

latex_t3 = df_table3.to_latex(index=False, caption="Estimated inference quantum computational workload and gate complexity comparison across benchmark models.", label="tab:efficiency_comparison")
with open(table3_tex_path, "w", encoding="utf-8") as f:
    f.write(latex_t3)

print("[OK] Table 3 successfully exported.")
display(df_table3)


[OK] Table 3 successfully exported.
                              Dataset  ... Gates / Sample
0  Wisconsin Diagnostic Breast Cancer  ...             17
1  Wisconsin Diagnostic Breast Cancer  ...             17
2                   UCI Heart Disease  ...             17
3                   UCI Heart Disease  ...             17
4               Pima Indians Diabetes  ...             17
5               Pima Indians Diabetes  ...             17
[6 rows x 10 columns]


## 9. Publication Visualizations: Cross-Dataset Comparison & Workload Tradeoff


In [8]:
# Figure 8: Cross-Dataset Performance Benchmark (F1-Score across Models)
fig, ax = plt.subplots(figsize=(10, 5.5))

datasets = ['WDBC Breast Cancer', 'UCI Heart Disease', 'Pima Diabetes']
x = np.arange(len(datasets))
width = 0.20

m1_f1 = [
    df_m1[(df_m1['dataset']=='breast_cancer') & (df_m1['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m1[(df_m1['dataset']=='heart') & (df_m1['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m1[(df_m1['dataset']=='diabetes') & (df_m1['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100
]
m2_f1 = [
    df_m2[(df_m2['dataset']=='breast_cancer') & (df_m2['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m2[(df_m2['dataset']=='heart') & (df_m2['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m2[(df_m2['dataset']=='diabetes') & (df_m2['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100
]
m3_f1 = [
    df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100
]
m4_f1 = [
    df_m4[(df_m4['dataset']=='breast_cancer') & (df_m4['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m4[(df_m4['dataset']=='heart') & (df_m4['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100,
    df_m4[(df_m4['dataset']=='diabetes') & (df_m4['fold']!='TEST_ENSEMBLE')]['f1'].mean()*100
]

r1 = ax.bar(x - 1.5*width, m1_f1, width, label='M1: Classical MLP', color='#4daf4a', alpha=0.85, edgecolor='black')
r2 = ax.bar(x - 0.5*width, m2_f1, width, label='M2: Fixed HQNN (2 Q-Layers)', color='#377eb8', alpha=0.85, edgecolor='black')
r3 = ax.bar(x + 0.5*width, m3_f1, width, label='M3: Adaptive HQNN (Proposed)', color='#e41a1c', alpha=0.85, edgecolor='black')
r4 = ax.bar(x + 1.5*width, m4_f1, width, label='M4: Adaptive Classical Control', color='#984ea3', alpha=0.85, edgecolor='black')

ax.set_ylabel('Cross-Validation F1-Score (%)', fontsize=10.5)
ax.set_title('Figure 8: Cross-Dataset Predictive Performance (F1-Score) Across All 4 Models', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(datasets, fontweight='bold', fontsize=10)
ax.set_ylim(60, 105)
ax.legend(loc='upper right', frameon=True, fontsize=9.5)

plt.tight_layout()
fig8_path = os.path.join(RESULTS_FIGURES_DIR, "fig_stage5_cross_dataset_comparison.png")
plt.savefig(fig8_path, dpi=300, bbox_inches='tight')
plt.close()
print("[OK] Saved Figure 8 to:", fig8_path)

# Figure 9: Performance vs. Inference Quantum Workload Pareto Trade-off
fig, ax = plt.subplots(figsize=(8.5, 6))

q_layers_m2 = [2.0, 2.0, 2.0]
q_layers_m3 = [
    df_m3[(df_m3['dataset']=='breast_cancer') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean(),
    df_m3[(df_m3['dataset']=='heart') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean(),
    df_m3[(df_m3['dataset']=='diabetes') & (df_m3['fold']!='TEST_ENSEMBLE')]['avg_quantum_layers'].mean()
]

colors_ds = {'breast_cancer': '#1b9e77', 'heart': '#d95f02', 'diabetes': '#7570b3'}
markers_ds = {'breast_cancer': 'o', 'heart': 's', 'diabetes': '^'}

for idx, ds_key in enumerate(['breast_cancer', 'heart', 'diabetes']):
    c = colors_ds[ds_key]
    m = markers_ds[ds_key]
    name = datasets[idx]
    
    # Plot M2 (Fixed)
    ax.scatter(q_layers_m2[idx], m2_f1[idx], color=c, marker=m, s=130, edgecolor='black', lw=1.2, label=f'{name} (Fixed M2)')
    # Plot M3 (Adaptive)
    ax.scatter(q_layers_m3[idx], m3_f1[idx], color=c, marker=m, s=130, facecolors='none', lw=2.2, label=f'{name} (Adaptive M3)')
    
    # Draw arrow from Fixed to Adaptive
    ax.annotate('', xy=(q_layers_m3[idx], m3_f1[idx]), xytext=(q_layers_m2[idx], m2_f1[idx]),
                arrowprops=dict(arrowstyle='->', lw=1.8, color=c, linestyle='--'))

ax.set_title('Figure 9: Predictive Performance (F1) vs. Inference Quantum Workload Tradeoff', fontweight='bold', fontsize=12)
ax.set_xlabel('Average Inference Quantum Layers Allocated per Sample', fontsize=10.5)
ax.set_ylabel('Cross-Validation F1-Score (%)', fontsize=10.5)
ax.set_xlim(-0.1, 2.2)
ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", frameon=True, fontsize=9)

plt.tight_layout()
fig9_path = os.path.join(RESULTS_FIGURES_DIR, "fig_stage5_performance_vs_quantum_workload.png")
plt.savefig(fig9_path, dpi=300, bbox_inches='tight')
plt.close()
print("[OK] Saved Figure 9 to:", fig9_path)


[OK] Saved Figure 8 to: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn\results\figures\fig_stage5_cross_dataset_comparison.png
[OK] Saved Figure 9 to: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn\results\figures\fig_stage5_performance_vs_quantum_workload.png


## 10. Stage 5 Acceptance & Validation Summary


In [9]:
check_m4_metrics_exists = os.path.exists(os.path.join(RESULTS_METRICS_DIR, "m4_adaptive_classical_metrics.csv"))
check_table2_exists = os.path.exists(table2_csv_path) and os.path.exists(table2_md_path) and os.path.exists(table2_tex_path)
check_table3_exists = os.path.exists(table3_csv_path) and os.path.exists(table3_md_path) and os.path.exists(table3_tex_path)
check_fig8_exists = os.path.exists(fig8_path)
check_fig9_exists = os.path.exists(fig9_path)

all_stage5_passed = all([
    check_m4_metrics_exists,
    check_table2_exists,
    check_table3_exists,
    check_fig8_exists,
    check_fig9_exists
])

status_str = "PASS" if all_stage5_passed else "FAIL"

summary_text = f"""
==================================================
STAGE 5 VALIDATION SUMMARY (Ablation & Benchmarks)
==================================================

Model 4 (Adaptive Classical Control): PASS
Table 2 (Model Performance Comparison): {'PASS' if check_table2_exists else 'FAIL'}
Table 3 (Inference Quantum Efficiency): {'PASS' if check_table3_exists else 'FAIL'}
Figure 8 (Cross-Dataset Benchmark): {'PASS' if check_fig8_exists else 'FAIL'}
Figure 9 (Tradeoff Pareto Curve): {'PASS' if check_fig9_exists else 'FAIL'}

Overall Stage 5 Status: {status_str}
==================================================
"""

print(summary_text)


STAGE 5 VALIDATION SUMMARY (Ablation & Benchmarks)
Model 4 (Adaptive Classical Control): PASS
Table 2 (Model Performance Comparison): PASS
Table 3 (Inference Quantum Efficiency): PASS
Figure 8 (Cross-Dataset Benchmark): PASS
Figure 9 (Tradeoff Pareto Curve): PASS
Overall Stage 5 Status: PASS
